# Lesson 03 — Images → Captions

Before we can search a collection of images, we need to turn each one into a
natural-language **caption** using an image-captioning model (BLIP).

This lesson:
1. Uploads a folder of sample images to S3
2. Runs a Batch job that captions each image on the GPU and saves the results back to S3
3. Downloads a few images and shows them here next to their generated captions

### What is image captioning?
Image captioning models look at a photo and generate a sentence describing it, e.g.
`"a dog playing in a park"`. This turns pixels into searchable text.

## Step 1 — Check .env

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../../.env")

S3_BUCKET = os.environ["S3_BUCKET"]
print(f"S3 bucket : {S3_BUCKET}")
print(f"Job queue : {os.environ['BATCH_JOB_QUEUE']}")

## Step 2 — Upload images & submit Batch job

`submit_job.py` does two things:
- Uploads every image in `assets/images/` to `s3://<bucket>/images/sample/`
- Submits the captioning job to Batch, passing `S3_BUCKET` and `IMAGE_PREFIX` as env vars

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "submit_job.py", "--images-dir", "assets/images", "--batch-stem", "sample"],
)
print("Exit code:", result.returncode)

## Step 3 — Download the caption manifest from S3

In [ ]:
import json

import boto3

s3  = boto3.client("s3")
key = "captions/sample/manifest.json"

obj = s3.get_object(Bucket=S3_BUCKET, Key=key)
manifest = json.loads(obj["Body"].read())

print(f"Found {len(manifest)} captions (showing first 5):")
for entry in manifest[:5]:
    print(f"  {entry['image_key']}  →  {entry['caption']!r}")

## Step 4 — Display images next to their generated captions

In [ ]:
import io
import os
import matplotlib.pyplot as plt
from PIL import Image

# Show up to 6 captioned images
to_show   = manifest[:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes      = axes.flatten()

for ax, entry in zip(axes, to_show):
    obj  = s3.get_object(Bucket=S3_BUCKET, Key=entry["image_key"])
    img  = Image.open(io.BytesIO(obj["Body"].read()))
    ax.imshow(img)
    ax.set_title(entry["caption"], fontsize=9, wrap=True)
    ax.axis("off")

plt.suptitle("Images captioned by BLIP", fontsize=12)
plt.tight_layout()
plt.show()

## Key Takeaway

> The S3 ↔ Batch pattern: **download input from S3 → process → upload results to S3**.
> Every Batch job in this course follows this pattern. Learn it once, use it everywhere.

---

## Next lesson → [04 — Captions → Embeddings](../04-captions-to-embeddings/notebook.ipynb)

We'll embed each caption with CLIP's text encoder to turn text into vectors — this is where the GPU does real work.